# MNISymm_coreg_normalize_pipeline

This is an "indirect" pipeline, in that each image is coregistered *before* being normalized to template space.

**Pipeline**:

- coregister T1 anatomical (entire brain + extra-brain tissue) image to reference

- for each subject-week, get a cerebellar isolation mask (this is its own thing - can be used in others. So have this in its own notebook)

- for each subject-week, get normalization files for MNISymm template space

- for each subject-week, normalize into MNI (symmetric):
    
    - white matter segmentation

    - grey matter segmentation

    - T1 anatomical

**NOTE**: this normalization will be done with respect to each week (i.e. reslice using the deformation file from each individual week, NOT the reference week).

## MNISymm_coreg_normalize

## ROUGH readme

**Pipeline**

- [x] native space images --> whole-image coregistration to reference image (where *reference* is the first image available from that subject)
- [ ] (rerun) isolate cerebellum
- [ ] (rerun) Using the coregistered images: create normalization files for MNI symmetric template
- [ ] normalize to the MNI symmetric template (using `reslice`) (for each subject-week):
    - [ ] white matter segmentations
    - [ ] grey matter segmentations
    - [ ] T1 anatomicals 


### NOTE

this pipeline uses the updated version of SUITPy (from master branch; master branch up-to-date with developer branch).

As of June 15 (at 6:39pm), used updated SUITPy for isolation masks.

1. Coregister each subject's T1 anatomical to their reference week - take their first measurement week as the reference week (SPM: use `sc_anat`).

2. Using the cerebellar isolation mask for each subject-week, get their normalization files in MNISymm space.

3. Normalize (T1_anat, wm, gm segmentations) into symmetric space (for each subject-week)

In [1]:
# need path to root directory
import sys
sys.path.append('/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/')

In [2]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
import ants

import SUITPy as suit
import SUITPy.atlas as atlas

import nitools as nt
from image_processing import tissue_extractor as te
from image_processing import avg_vol as av

from pathlib import Path
import os

In [3]:
# directories
base_dir = '/cifs/diedrichsen/data/smarts_cerebellum'
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

## Get normalization files for each subject-week

In [ ]:
"""
# write normalization files for each subject-week (full-image coregistered)
# this just takes the t1_anatomicals and the isolation mask for each subject.


#_______________________________
# base loop
for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    # files required for creating subject-week normalization files: T1 anatomical, (binary) isolation mask
    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    mask_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'



    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue

    if not Path(mask_path).is_file():
        print(f'mask path does not exist for {subj_id} in week {week}')
        continue
    

    # this is a new folder for each subject-week
    results_path = Path(base_dir)/'MNISym/full_img_coreg'/subj_id/week
    results_path.mkdir(parents=True, exist_ok = True)
    #__________________________________

    # function goes here

    te.normalize(t1_path, mask_path, results_path, space = 'MNI152NLin2009cSymC')

    print(f'{subj_id} {week} normalization done \n')


# store in anat_dir/MNISym/full_img_coreg/subj_id/week for each subject-week
"""

Normalizing CU_2310_W0_T1 to tpl-MNI152NLin2009cSymC_T1w.nii.gz
Saving the normalized image into CU_2310_W0_T1_space-MNI152NLin2009cSymC.nii.gz
Saving deformation field into CU_2310_W0_T1_to-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving inverse deformation field into CU_2310_W0_T1_from-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving the Jacobian determinant to CU_2310_W0_T1_to-MNI152NLin2009cSymC_mode-image_detJ.nii.gz
Saving the log-Jacobian determinant to CU_2310_W0_T1_to-MNI152NLin2009cSymC_mode-image_log_detJ.nii.gz
CU_2310 W0 normalization done 

Normalizing CU_2310_W4_T1 to tpl-MNI152NLin2009cSymC_T1w.nii.gz
Saving the normalized image into CU_2310_W4_T1_space-MNI152NLin2009cSymC.nii.gz
Saving deformation field into CU_2310_W4_T1_to-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving inverse deformation field into CU_2310_W4_T1_from-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving the Jacobian determinant to CU_2310_W4_T1_to-MNI152NLin2009cSymC_mode-image_detJ.nii.gz
Saving the 

### Function for reslice loop (this is temporarily in lieu of a proper subject-loop function)

should re-check reslice loop: changed path to have all subj files in one folder instead of subj_week

All good

In [ ]:
"""
def reslice_loop(
        sub_dir, # e.g. MNISymm_T1
        suffix, # name of normalized image
        tissue = None
        ):
    
    tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c3'
    }

    # base loop _____________________________________
    for i in range(0, p_df.shape[0]):
        p_id = p_df['ID'].iloc[i]
        week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
        p_centre = (str(p_df['Centre'].iloc[i])).strip()
        
        subj_id = f'{p_centre.strip()}_{p_id}'

        # required files for reslice
        if not tissue == None:
            img_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'
        else:
            img_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
        print(f'using {img_path}')

        


        mask_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'
        fwd_def = f'{base_dir}/MNISym/full_img_coreg/{subj_id}/{week}/{subj_id}_{week}_T1_to-MNI152NLin2009cSymC_mode-image_xfm.nii.gz'

        # check that paths exist
        if not Path(img_path).is_file():
            print(f'T1 path does not exist for {subj_id} in week {week}')
            continue
            
        if not Path(mask_path).is_file():
            print(f'mask path does not exist for {subj_id} in week {week}')
            continue

        if not Path(fwd_def).is_file():
            print(f'fwd def path does not exist for {subj_id} in week {week}')
            continue
      
            
        # CHECK THIS PART using one subject
        results_path = Path(base_dir)/sub_dir/subj_id
        results_path.mkdir(parents=True, exist_ok = True)
      
        # we will use the week option in reslice, so it will save each week's resliced image to each week's directory
        te.reslice(img_path = img_path,
                   fwd_def = fwd_def,
                   mask_path = mask_path,

                   results_path = results_path,
                   subj_id = subj_id,
                   suffix = suffix,
                   week = week)
        
        if not tissue == None:
            print(f'Normalization done for {subj_id} {week} for {tissue}')
        else:
            print(f'Normalization done for {subj_id} at {week} for T1 anatomical')

"""

When reslicing segmentation (tissue) files: suffix = wm_MNISymm
When reslicing T1 anatomical files: suffix = T1_MNISymm

("normalized" is already included as the actual suffix for the image, so this would be {suffix}_normalized)

So we have three folders: MNISymm_<type> where type = T1, wm, gm

In each of these folders, we will save the resliced t1 anatomical or tissue file for each subject-week.

So it will have: subject --> week --> normalized file (for each of these three folders) - so one file per subject week.

We can probably collapse this into just subjects and have all of their week files in the same directory.

## Normalize the images to template (MNISymm) space

In [ ]:
"""
# normalize T1 anatomicals to MNISymm template space
reslice_loop(sub_dir = f'MNISym_T1',
             suffix = 'MNISym_T1')
"""

using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W0/CU_2310_W0_T1.nii
Normalization done for CU_2310 at W0 for T1 anatomical
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W4/CU_2310_W4_T1.nii
Normalization done for CU_2310 at W4 for T1 anatomical
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W12/CU_2310_W12_T1.nii
Normalization done for CU_2310 at W12 for T1 anatomical
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W24/CU_2310_W24_T1.nii
Normalization done for CU_2310 at W24 for T1 anatomical
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W52/CU_2310_W52_T1.nii
Normalization done for CU_2310 at W52 for T1 anatomical
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W0/CU_2538_W0_T1.nii
Normalization done for CU_2538 at W0 for T1 anatomical
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W4/CU_2538_W4_T1.nii
Normalization done for CU_2538 at W4 for T1 a

In [ ]:
"""
# normalize wm segmentations to MNISymm template space
reslice_loop(sub_dir = f'MNISym_WM',
             suffix = 'MNISym_WM',
             tissue = 'wm')
"""

using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W0/c2CU_2310_W0_T1.nii
Normalization done for CU_2310 W0 for wm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W4/c2CU_2310_W4_T1.nii
Normalization done for CU_2310 W4 for wm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W12/c2CU_2310_W12_T1.nii
Normalization done for CU_2310 W12 for wm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W24/c2CU_2310_W24_T1.nii
Normalization done for CU_2310 W24 for wm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W52/c2CU_2310_W52_T1.nii
Normalization done for CU_2310 W52 for wm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W0/c2CU_2538_W0_T1.nii
Normalization done for CU_2538 W0 for wm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W4/c2CU_2538_W4_T1.nii
Normalization done for CU_2538 W4 for wm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2663/W0/c2CU

In [ ]:
"""
# normalize gm segmentations to MNISymm template space
reslice_loop(sub_dir = f'MNISym_GM',
             suffix = 'MNISym_GM',
             tissue = 'gm')
"""

using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W0/c1CU_2310_W0_T1.nii
Normalization done for CU_2310 W0 for gm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W4/c1CU_2310_W4_T1.nii
Normalization done for CU_2310 W4 for gm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W12/c1CU_2310_W12_T1.nii
Normalization done for CU_2310 W12 for gm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W24/c1CU_2310_W24_T1.nii
Normalization done for CU_2310 W24 for gm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2310/W52/c1CU_2310_W52_T1.nii
Normalization done for CU_2310 W52 for gm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W0/c1CU_2538_W0_T1.nii
Normalization done for CU_2538 W0 for gm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2538/W4/c1CU_2538_W4_T1.nii
Normalization done for CU_2538 W4 for gm
using /cifs/diedrichsen/data/smarts_cerebellum/anatomicals/CU_2663/W0/c1CU

## Run regression on normalized T1 anats, wm and gm segmentations

Regression on wm segmentation normalized to MNISym template.

**Before doing this** on GM, we renamed the files from: (since these files were coregistered (whole-image), and used this new affine for normalization, so should reflect that)
    
prev: `{subj_id}_{week}_MNISym_GM_reslice.nii.gz`
    
new: `{subj_id}_{week}_MNISym_GM_coreg_reslice.nii.gz`

For now, going to save the regression images (intercept and slope) in the same directory as the resliced images.

### Renaming (GM) files from reslice

In [ ]:
# #renaming MNISym_GM_reslice files to add "coreg" (whole-image coregistration normalization)

# from helper_functions import rename_files

# for i in range(0, p_df.shape[0]):

#     p_id = p_df['ID'].iloc[i]
#     week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
#     p_centre = (str(p_df['Centre'].iloc[i])).strip()

#     subj_id = f'{p_centre.strip()}_{p_id}'

#     # directory for each subject's files (each subject gets their own base_dir here)
#     mni_base_dir = f'/cifs/diedrichsen/data/smarts_cerebellum/MNISym_GM/{subj_id}'

#     old_path = f'{mni_base_dir}/{subj_id}_{week}_MNISym_GM_reslice.nii.gz'
#     new_path = f'{mni_base_dir}/{subj_id}_{week}_MNISym_GM_coreg_reslice.nii.gz'

#     rename_files.rename_file(old_path = old_path, new_path = new_path)

## Regression

In [6]:
# regression on wm segemnetations in MNISym template space.
for subj in p_df['subj_id'].unique():

    # find each subject's reference image, and run it through the regression
    refT1 = (p_df.loc[(p_df['subj_id']==subj), 'RefT1'].iloc[0]).strip()
    ref_img = f'{base_dir}/MNISym_GM/{subj}/{subj}_{refT1}_MNISym_GM_coreg_reslice.nii.gz'

    betas, X, Y, intercept_img, slope_img = av.avg_vol(subj_id = subj,
                         reference_img = ref_img,
                         results_path = f'{base_dir}/MNISym_GM/{subj}',
                         image_suffix = 'MNISym_GM_coreg_reslice', # name will be like <subj_id>_MNISym_GM_reslice_<alg = intercept/slope>.nii.gz

                         )
     

currently on /cifs/diedrichsen/data/smarts_cerebellum/MNISym_GM/CU_2697/CU_2697_W4_MNISym_GM_coreg_reslice.nii.gz
skipping W0
skipping W12
skipping W24
skipping W52
skipping CU_2697: insufficient measurement weeks
